# Front-of-Package Nutrition Label RCT Analysis (Grummon et al., 2025 Benchmark)

In [32]:
import importlib
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch

import grummon2025_ols

importlib.reload(grummon2025_ols)
from grummon2025_ols import (
    AUDIT_PREFIX,
    MODEL_PALETTES,
    NEUTRALS,
    PERSONA_LABELS,
    build_grummon2025_coef_table,
    build_grummon2025_summary_table,
    evaluate_grummon2025_metrics,
    fit_grummon2025_ols,
    format_entry_display_label,
    format_raw_model_name,
    load_all_model_regressions,
    load_grummon2025_rct,
    load_single_attr_ce_table,
    parse_entry_metadata,
    set_academic_palette,
    subsample_grummon2025_participants,
)


## 1. Design

In [33]:
# Evaluation subsample (must match the collection notebook)
N_PARTICIPANTS = 200
SUBSAMPLE_SEED = 42

# Kev checkpoints (record `model` field), in table order; each needs a run for every PERSONA_MODES mode in every run-set folder
MODEL_IDS = [
    "jaredpalmer/kev-0.8b",
    "kev-0.8b-resp-issp+twin2k-grouped-newvalid/best",
    "jaredpalmer/kev-4b",
    "kev-4b-resp-issp+twin2k-grouped-newvalid/best",
    "jaredpalmer/kev-9b",
    "kev-9b-resp-issp+twin2k-grouped-newvalid/best",
]

# Persona modes reported in the tables and error figures ("random" = no persona, "persona" = respondent persona)
PERSONA_MODES = ["persona"]

# Persona attributes of the collection, split into seen / unseen in survey training
SEEN_COLS = ("SEX", "AGE", "EDULEVEL", "US_INC", "RACE", "HHSIZE")
UNSEEN_COLS = ("HISPANIC", "CHILDREN", "SNAP", "WIC", "NFPUSE", "ONLINEAMT", "HEALTHLIT")

# Kev run sets: (label prefix, folder, seen attributes, unseen attributes, prompt source: "original" or "audited").
# More sets can be listed side by side; their prefixes must differ and each be "" or AUDIT_PREFIX.
KEV_RUN_SETS = [
    ("", "data-audit", SEEN_COLS, UNSEEN_COLS, "audited"),
]

# Main-text figure: run set (by label prefix) and (off-the-shelf, fine-tuned) display names without prefix
FIG_KEV_PREFIX = ""
FIG_MODELS = ("Kev 9B (base)", "Kev 9B (FT)")

# Panel B axis labels by attribute code ("*" is added for unseen attributes)
ATTR_SHORT_LABELS = {
    "SEX": "Sex",
    "AGE": "Age",
    "EDULEVEL": "Education",
    "US_INC": "Household income",
    "RACE": "Race",
    "HHSIZE": "Household size",
    "HISPANIC": "Hispanic ethnicity",
    "CHILDREN": "Household children",
    "SNAP": "SNAP participation",
    "WIC": "WIC participation",
    "NFPUSE": "Nutrition Facts usage",
    "ONLINEAMT": "Online grocery share",
    "HEALTHLIT": "Nutrition literacy",
}

## 2. Human ground truth and subsample

In [34]:
raw_df = load_grummon2025_rct()
human_ols_df = fit_grummon2025_ols(raw_df)
beta_star = human_ols_df["beta"].copy()

sub_df = subsample_grummon2025_participants(raw_df, n_participants=N_PARTICIPANTS, seed=SUBSAMPLE_SEED)
human_sub_ols_df = fit_grummon2025_ols(sub_df)
beta_human_sub = human_sub_ols_df["beta"].copy()
human_sub_metrics = evaluate_grummon2025_metrics(beta_human_sub, beta_star)

# Sample sizes used in table/figure labels and notes
n_full_participants, n_full_tasks = raw_df["CASEID"].nunique(), len(raw_df)
n_sub_participants, n_sub_tasks, d_arms = sub_df["CASEID"].nunique(), len(sub_df), len(beta_star)

print(
    f"Full sample: {n_full_participants:,} participants, {n_full_tasks:,} tasks, {d_arms} treatment effects | "
    f"Subsample: {n_sub_participants:,} participants, {n_sub_tasks:,} tasks | Subsample vs. full human: "
    f"MAPE = {human_sub_metrics['MAPE (%)']:.2f}%, MSE = {human_sub_metrics['MSE']:.4f}, "
    f"Spearman rho = {human_sub_metrics['Spearman rho']:+.2f}"
)
human_sub_ols_df.assign(full_human_beta=beta_star, diff=beta_human_sub - beta_star).round(4)

Full sample: 5,633 participants, 33,798 tasks, 5 treatment effects | Subsample: 200 participants, 1,200 tasks | Subsample vs. full human: MAPE = 22.66%, MSE = 0.0016, Spearman rho = +0.40


,arm,beta,se,z,p_value,full_human_beta,diff
label,,,,,,,
Nutrition Info label (vs. Positive),NutritionInfo,0.1031,0.0759,1.3585,0.1743,0.1844,-0.0813
High In warning label (vs. Positive),HighIn,0.0365,0.0692,0.5267,0.5984,0.0541,-0.0176
Positive + Nutrition Info (vs. Positive),Positive+NutritionInfo,0.2031,0.0655,3.1006,0.0019,0.1788,0.0243
Positive + High In (vs. Positive),Positive+HighIn,0.1317,0.0671,1.9618,0.0498,0.1194,0.0123
Spectrum 1-to-5 star label (vs. Positive),Spectrum,0.1222,0.0696,1.7565,0.0790,0.1083,0.0139


## 3. Kev runs: HC1 OLS and choice cross-entropy

In [35]:
prefixes = [run_set[0] for run_set in KEV_RUN_SETS]
if len(set(prefixes)) != len(prefixes) or not set(prefixes) <= {"", AUDIT_PREFIX}:
    raise ValueError(f"Run-set prefixes must be distinct and each '' or {AUDIT_PREFIX!r}, got {prefixes}")

model_ols_dfs, ce_stats, ce_stats_by_prefix = {}, {}, {}
for prefix, folder, _, _, _ in KEV_RUN_SETS:
    set_ols, set_ce = load_all_model_regressions(
        sub_df, kev_data_dir=folder, model_ids=MODEL_IDS, persona_modes=PERSONA_MODES, kev_label_prefix=prefix
    )
    if set(set_ols) & set(model_ols_dfs):
        raise ValueError(f"Labels collide across run sets: {sorted(set(set_ols) & set(model_ols_dfs))}")
    model_ols_dfs.update(set_ols)
    ce_stats.update(set_ce)
    ce_stats_by_prefix[prefix] = set_ce

print(f"Loaded and fitted {len(model_ols_dfs)} Kev model datasets:")
for name in model_ols_dfs:
    info = ce_stats[name]
    print(f" - {name}: T={info['temperature']:.2f}, CE={info['ce_nats']:.4f} nats ({info['ce_bits']:.4f} bits)")

Loaded and fitted 6 Kev model datasets:
 - jaredpalmer/kev-0.8b (Respondent Persona, N=200, n=1200): T=2.35, CE=1.0237 nats (1.4770 bits)
 - kev-0.8b-resp-issp+twin2k-grouped-newvalid/best (Respondent Persona, N=200, n=1200): T=1.00, CE=1.0769 nats (1.5536 bits)
 - jaredpalmer/kev-4b (Respondent Persona, N=200, n=1200): T=2.41, CE=1.0483 nats (1.5124 bits)
 - kev-4b-resp-issp+twin2k-grouped-newvalid/best (Respondent Persona, N=200, n=1200): T=1.00, CE=1.0161 nats (1.4659 bits)
 - jaredpalmer/kev-9b (Respondent Persona, N=200, n=1200): T=2.19, CE=1.0678 nats (1.5405 bits)
 - kev-9b-resp-issp+twin2k-grouped-newvalid/best (Respondent Persona, N=200, n=1200): T=1.00, CE=0.9343 nats (1.3479 bits)


In [36]:
human_sub_key = f"Human Subsample Baseline (N={n_sub_participants}, n={n_sub_tasks})"
estimators_beta = {
    human_sub_key: beta_human_sub,
    **{name: df_m["beta"] for name, df_m in model_ols_dfs.items()},
}

comparison_rows = {}
for name, b_hat in estimators_beta.items():
    vs_full = evaluate_grummon2025_metrics(b_hat, beta_star)
    vs_sub = evaluate_grummon2025_metrics(b_hat, beta_human_sub)
    comparison_rows[name] = {
        "MAPE vs. Full Human (%)": vs_full["MAPE (%)"],
        "MSE vs. Full Human": vs_full["MSE"],
        "Spearman rho vs. Full Human": vs_full["Spearman rho"],
        "MAPE vs. Human Subsample (%)": vs_sub["MAPE (%)"],
        "MSE vs. Human Subsample": vs_sub["MSE"],
    }

comparison_metrics_df = pd.DataFrame(comparison_rows).T.sort_values("MSE vs. Full Human", ascending=True)
summary_table_df = build_grummon2025_summary_table(
    comparison_metrics_df,
    ce_stats,
    human_sub_key=human_sub_key,
    persona_modes=PERSONA_MODES,
    n_sub_participants=n_sub_participants,
    n_sub_tasks=n_sub_tasks,
)
coef_table_df = build_grummon2025_coef_table(
    human_ols_df,
    human_sub_ols_df,
    model_ols_dfs,
    persona_mode="Respondent Persona",
    n_full_participants=n_full_participants,
    n_sub_participants=n_sub_participants,
)

single_attr_by_prefix = {
    prefix: load_single_attr_ce_table(
        sub_df, ablation_dir=f"{folder}/persona_ablation", model_ids=MODEL_IDS,
        seen_cols=seen, unseen_cols=unseen, label_prefix=prefix,
    )
    for prefix, folder, seen, unseen, _ in KEV_RUN_SETS
}
attr_keys = ["Attribute group", "Code", "Shopper attribute"]
single_attr_ce_df = single_attr_by_prefix[prefixes[0]]
for prefix in prefixes[1:]:
    single_attr_ce_df = single_attr_ce_df.merge(single_attr_by_prefix[prefix], on=attr_keys, how="outer", sort=False)
if single_attr_ce_df["Code"].duplicated().any():
    raise ValueError("An attribute is grouped as seen in one run set and unseen in another")
# Row order: attributes in declared order, starting from the run set with the most attributes
attr_order = list(dict.fromkeys(
    c for _, _, seen, unseen, _ in sorted(KEV_RUN_SETS, key=lambda rs: -(len(rs[2]) + len(rs[3]))) for c in (*seen, *unseen)
))
single_attr_ce_df = single_attr_ce_df.set_index("Code").loc[attr_order].reset_index()[single_attr_ce_df.columns]

print("=== Summary table ===")
display(summary_table_df)
print("\n=== Arm-level HC1 OLS coefficients (respondent persona) ===")
display(coef_table_df)
coverage = ", ".join(f"{prefix or '(no prefix)'} {len(seen)}+{len(unseen)}" for prefix, _, seen, unseen, _ in KEV_RUN_SETS)
print(f"\n=== Single-Attribute Choice Cross-Entropy (bits), seen+unseen attributes per run set: {coverage} ===")
display(single_attr_ce_df.round(4))

=== Summary table ===


,Estimator,CE (bits),MAPE (%),MSE,Spearman rho
0,"Human subsample (N = 200, n = 1,200)",—,22.7%,0.0016,+0.40
1,Kev 0.8B (base),1.4770,58.5%,0.0074,+0.50
2,Kev 0.8B FT,1.5536,66.5%,0.0078,+0.70
3,Kev 4B (base),1.5124,95.9%,0.0088,+0.80
4,Kev 4B FT,1.4659,83.0%,0.0099,+0.80
5,Kev 9B (base),1.5405,63.7%,0.0047,+0.80
6,Kev 9B FT,1.3479,44.3%,0.0022,+0.90



=== Arm-level HC1 OLS coefficients (respondent persona) ===


,Trial arm (vs. Positive),"Full human (N = 5,633)",Human subsample (N = 200),Kev 0.8B (base),Kev 0.8B FT,Kev 4B (base),Kev 4B FT,Kev 9B (base),Kev 9B FT
0,Nutrition Info label (vs. Positive),+0.184 (0.012),+0.103 (0.076),+0.076 (0.002),+0.116 (0.003),+0.124 (0.003),+0.065 (0.004),+0.224 (0.005),+0.177 (0.003)
1,Positive + Nutrition Info (vs. Positive),+0.179 (0.012),+0.203 (0.066),+0.094 (0.002),+0.105 (0.003),+0.146 (0.003),+0.069 (0.004),+0.229 (0.004),+0.168 (0.004)
2,Positive + High In (vs. Positive),+0.119 (0.012),+0.132 (0.067),+0.011 (0.002),-0.038 (0.003),+0.000 (0.003),-0.002 (0.004),+0.023 (0.005),+0.042 (0.003)
3,Spectrum 1-to-5 star label (vs. Positive),+0.108 (0.012),+0.122 (0.070),+0.032 (0.003),+0.057 (0.003),+0.038 (0.003),+0.057 (0.004),+0.109 (0.005),+0.139 (0.003)
4,High In warning label (vs. Positive),+0.054 (0.012),+0.036 (0.069),+0.067 (0.002),+0.014 (0.003),-0.088 (0.003),-0.022 (0.004),-0.047 (0.005),-0.010 (0.003)



=== Single-Attribute Choice Cross-Entropy (bits), seen+unseen attributes per run set: (no prefix) 6+7 ===


,Attribute group,Code,Shopper attribute,Kev 0.8B (base),Kev 0.8B FT,Kev 4B (base),Kev 4B FT,Kev 9B (base),Kev 9B FT
0,Seen in training,SEX,Sex,1.5253,1.5472,1.3948,1.4341,1.4902,1.3468
1,Seen in training,AGE,Age,1.4806,1.5426,1.3974,1.4366,1.5011,1.3532
2,Seen in training,EDULEVEL,Education,1.5152,1.5475,1.3897,1.4267,1.5375,1.3478
3,Seen in training,US_INC,Income,1.4764,1.5214,1.4416,1.4362,1.4908,1.3462
4,Seen in training,RACE,Race,1.4746,1.5389,1.4243,1.4258,1.5054,1.3437
5,Seen in training,HHSIZE,"How many people, including yourself, live in y...",1.5395,1.5477,1.5033,1.4343,1.3946,1.3486
6,Unseen in training,HISPANIC,Hispanic or Latino,1.5420,1.5527,1.3834,1.4314,1.5001,1.3497
7,Unseen in training,CHILDREN,How many children (ages 0-18) currently live i...,1.5658,1.5499,1.5430,1.4370,1.3935,1.3420
8,Unseen in training,SNAP,SNAP / Food Stamps recipient,1.5709,1.5419,1.4581,1.4295,1.5669,1.3507
9,Unseen in training,WIC,Do you or anyone else in your household curren...,1.4839,1.5286,1.5181,1.4360,1.4193,1.3404


## 4. Paste-ready tables

In [37]:
def to_paper_md(df: pd.DataFrame) -> str:
    """Compact pipe table (text columns left, numbers right) with italic n/N, matching manuscript.qmd."""
    text_cols = {"Estimator", "Trial arm (vs. Positive)", "Attribute group", "Shopper attribute"}
    rows = [list(df.columns)] + [[str(v) for v in r] for r in df.itertuples(index=False)]
    lines = ["| " + " | ".join(r) + " |" for r in rows]
    lines.insert(1, "|" + "|".join(":---" if c in text_cols else "---:" for c in df.columns) + "|")
    md = "\n".join(lines)
    return md.replace("(N = ", "(*N* = ").replace("(n = ", "(*n* = ").replace(", n = ", ", *n* = ")


uniform_ce_bits = math.log2(3.0)
ref_arm = re.fullmatch(r".*\(vs\. (.+)\)", str(beta_star.index[0])).group(1)

# Inference temperature per checkpoint, identical across run sets
temps_by_model: dict[str, float] = {}
for prefix, set_ce in ce_stats_by_prefix.items():
    for key, info in set_ce.items():
        raw_model = str(parse_entry_metadata(key)["raw_model"]).removeprefix(prefix)
        if raw_model in temps_by_model and abs(temps_by_model[raw_model] - info["temperature"]) > 1e-9:
            raise ValueError(f"{raw_model}: temperature differs between run sets ({temps_by_model[raw_model]} vs {info['temperature']})")
        temps_by_model[raw_model] = float(info["temperature"])
temp_txt = "; ".join(f"`{format_raw_model_name(m)}` at $T = {temps_by_model[m]:.2f}$" for m in MODEL_IDS)

# Prompt source and persona size of each run set
PROMPT_SOURCE_TXT = {
    "original": "the original prompts",
    "audited": "prompts built from the audited survey items",
}


def run_set_rows(prefix: str) -> str:
    """How the notes refer to the rows/columns of one run set."""
    if prefix:
        return f"`{prefix}` Kev rows"
    return "Unprefixed Kev rows" if len(KEV_RUN_SETS) > 1 else "Kev rows"


prompt_txt = "; ".join(
    f"{run_set_rows(prefix)} use {PROMPT_SOURCE_TXT[source]} with {len(seen) + len(unseen)}-attribute respondent personas"
    for prefix, _, seen, unseen, source in KEV_RUN_SETS
)
prompt_txt = prompt_txt[0].upper() + prompt_txt[1:]
REGIME_TXT = {"random": "random-persona prompts", "persona": "respondent-persona conditioning"}
regime_txt = " and ".join(REGIME_TXT[m] for m in PERSONA_MODES)
sample_txt = f"*N* = {n_sub_participants:,} shoppers, *n* = {n_sub_tasks:,} choices"
full_txt = f"*N* = {n_full_participants:,} shoppers, {n_full_tasks:,} choices; @grummon2025nutrition"

# --- Main-text table: @tbl-grummon2025-ols ---
print(to_paper_md(summary_table_df.rename(columns={"Spearman rho": "Spearman $\\rho$"})))
print("\n: Choice cross-entropy and OLS treatment-effect recovery on the nutrition labeling randomized trial {#tbl-grummon2025-ols}\n")
print('::: {custom-style="Table Note"}')
print(
    f"*Note.* Three-option choice cross-entropy (CE, in bits; uniform baseline $\\log_2 3 = {uniform_ce_bits:.4f}$ bits) and participant-level "
    f"HC1 ordinary least squares (OLS) treatment-effect error across the $d = {d_arms}$ front-of-package labeling arms (relative to `{ref_arm}`) "
    f"under {regime_txt} on the evaluation subsample ({sample_txt}), benchmarked against full-sample human estimates $\\boldsymbol{{\\beta}}^*$ ({full_txt}). "
    f"Spearman rank correlation $\\rho$ is reported for respondent-persona OLS treatment effects against $\\boldsymbol{{\\beta}}^*$. "
    f"Decision models use {temp_txt}. {prompt_txt}."
)
print(":::\n\n")

| Estimator | CE (bits) | MAPE (%) | MSE | Spearman $\rho$ |
|:---|---:|---:|---:|---:|
| Human subsample (*N* = 200, *n* = 1,200) | — | 22.7% | 0.0016 | +0.40 |
| Kev 0.8B (base) | 1.4770 | 58.5% | 0.0074 | +0.50 |
| Kev 0.8B FT | 1.5536 | 66.5% | 0.0078 | +0.70 |
| Kev 4B (base) | 1.5124 | 95.9% | 0.0088 | +0.80 |
| Kev 4B FT | 1.4659 | 83.0% | 0.0099 | +0.80 |
| Kev 9B (base) | 1.5405 | 63.7% | 0.0047 | +0.80 |
| Kev 9B FT | 1.3479 | 44.3% | 0.0022 | +0.90 |

: Choice cross-entropy and OLS treatment-effect recovery on the nutrition labeling randomized trial {#tbl-grummon2025-ols}

::: {custom-style="Table Note"}
*Note.* Three-option choice cross-entropy (CE, in bits; uniform baseline $\log_2 3 = 1.5850$ bits) and participant-level HC1 ordinary least squares (OLS) treatment-effect error across the $d = 5$ front-of-package labeling arms (relative to `Positive`) under respondent-persona conditioning on the evaluation subsample (*N* = 200 shoppers, *n* = 1,200 choices), benchmarked agai

In [38]:
# --- Appendix table: @apptbl-grummon2025-coefs ---
print('::: {#apptbl-grummon2025-coefs size="scriptsize"}\n')
print(to_paper_md(coef_table_df))
print("\nParticipant-level OLS treatment-effect estimates on the nutrition labeling randomized trial\n")
print(":::\n")
print('::: {custom-style="Table Note"}')
print(
    f"*Note.* Ordinary least squares (OLS) point estimates and HC1 heteroskedasticity-robust standard errors (in parentheses) for the "
    f"{d_arms} front-of-package labeling arms relative to the `{ref_arm}` reference arm under respondent-persona conditioning ({sample_txt}), "
    f"alongside full-sample human estimates ({full_txt}). {prompt_txt}."
)
print(":::\n\n")

::: {#apptbl-grummon2025-coefs size="scriptsize"}

| Trial arm (vs. Positive) | Full human (*N* = 5,633) | Human subsample (*N* = 200) | Kev 0.8B (base) | Kev 0.8B FT | Kev 4B (base) | Kev 4B FT | Kev 9B (base) | Kev 9B FT |
|:---|---:|---:|---:|---:|---:|---:|---:|---:|
| Nutrition Info label (vs. Positive) | +0.184 (0.012) | +0.103 (0.076) | +0.076 (0.002) | +0.116 (0.003) | +0.124 (0.003) | +0.065 (0.004) | +0.224 (0.005) | +0.177 (0.003) |
| Positive + Nutrition Info (vs. Positive) | +0.179 (0.012) | +0.203 (0.066) | +0.094 (0.002) | +0.105 (0.003) | +0.146 (0.003) | +0.069 (0.004) | +0.229 (0.004) | +0.168 (0.004) |
| Positive + High In (vs. Positive) | +0.119 (0.012) | +0.132 (0.067) | +0.011 (0.002) | -0.038 (0.003) | +0.000 (0.003) | -0.002 (0.004) | +0.023 (0.005) | +0.042 (0.003) |
| Spectrum 1-to-5 star label (vs. Positive) | +0.108 (0.012) | +0.122 (0.070) | +0.032 (0.003) | +0.057 (0.003) | +0.038 (0.003) | +0.057 (0.004) | +0.109 (0.005) | +0.139 (0.003) |
| High In warni

In [39]:
# --- Appendix table: @apptbl-grummon2025-personas ---
# Model columns: each checkpoint followed by its counterparts in the other run sets
base_model_cols = [c.removeprefix(prefixes[0]) for c in single_attr_by_prefix[prefixes[0]].columns if c not in attr_keys]
model_cols = [prefix + c for c in base_model_cols for prefix in prefixes]
if set(model_cols) != set(single_attr_ce_df.columns) - set(attr_keys):
    raise ValueError(f"Unexpected single-attribute columns: {sorted(set(single_attr_ce_df.columns) - set(attr_keys))}")
persona_rows = []
for grp, label in (("Seen in training", "Seen"), ("Unseen in training", "Unseen")):
    grp_df = single_attr_ce_df[single_attr_ce_df["Attribute group"] == grp]
    for r in grp_df.to_dict("records"):
        persona_rows.append({
            "Attribute group": grp,
            "Shopper attribute": r["Shopper attribute"],
            **{c: ("—" if pd.isna(r[c]) else f"{r[c]:.4f}") for c in model_cols},
        })
    persona_rows.append({
        "Attribute group": f"**{label} mean**",
        "Shopper attribute": "—",
        **{c: f"**{grp_df[c].mean():.4f}** ({int(grp_df[c].notna().sum())})" for c in model_cols},
    })
persona_tbl_df = pd.DataFrame(persona_rows)
print('::: {#apptbl-grummon2025-personas size="scriptsize"}\n')
print(to_paper_md(persona_tbl_df))
print("\nSingle-attribute choice cross-entropy across seen and unseen shopper covariates\n")
print(":::\n")
print('::: {custom-style="Table Note"}')
print(
    f"*Note.* Three-option choice cross-entropy (in bits; uniform baseline $\\log_2 3 = {uniform_ce_bits:.4f}$ bits) on the evaluation subsample "
    f"({sample_txt}) of @grummon2025nutrition when conditioning each model on a single shopper attribute at a time, grouped into demographic "
    f"covariates present in the survey training corpus and food-shopping and nutrition covariates unseen during training. "
    + " ".join(
        f"{run_set_rows(prefix).replace(' rows', ' runs')} cover {len(seen)} seen and {len(unseen)} unseen attributes."
        for prefix, _, seen, unseen, _ in KEV_RUN_SETS
    )
    + (" — marks attributes not collected in a run set; group means average over the attributes each run covers (count in parentheses)."
       if len(KEV_RUN_SETS) > 1 else " Group means average over the attributes in each group (count in parentheses).")
)
print(":::")

::: {#apptbl-grummon2025-personas size="scriptsize"}

| Attribute group | Shopper attribute | Kev 0.8B (base) | Kev 0.8B FT | Kev 4B (base) | Kev 4B FT | Kev 9B (base) | Kev 9B FT |
|:---|:---|---:|---:|---:|---:|---:|---:|
| Seen in training | Sex | 1.5253 | 1.5472 | 1.3948 | 1.4341 | 1.4902 | 1.3468 |
| Seen in training | Age | 1.4806 | 1.5426 | 1.3974 | 1.4366 | 1.5011 | 1.3532 |
| Seen in training | Education | 1.5152 | 1.5475 | 1.3897 | 1.4267 | 1.5375 | 1.3478 |
| Seen in training | Income | 1.4764 | 1.5214 | 1.4416 | 1.4362 | 1.4908 | 1.3462 |
| Seen in training | Race | 1.4746 | 1.5389 | 1.4243 | 1.4258 | 1.5054 | 1.3437 |
| Seen in training | How many people, including yourself, live in your household? | 1.5395 | 1.5477 | 1.5033 | 1.4343 | 1.3946 | 1.3486 |
| **Seen mean** | — | **1.5019** (6) | **1.5409** (6) | **1.4252** (6) | **1.4323** (6) | **1.4866** (6) | **1.3477** (6) |
| Unseen in training | Hispanic or Latino | 1.5420 | 1.5527 | 1.3834 | 1.4314 | 1.5001 | 1.3497 |
|

## 5. Figures

In [ ]:
def plot_grummon2025_coefs_and_attributes(
    human_ols_df: pd.DataFrame,
    human_sub_ols_df: pd.DataFrame,
    model_ols_dfs: dict[str, pd.DataFrame],
    single_attr_ce_df: pd.DataFrame,
    palette: str = "nature",
    persona_mode: str = "Respondent Persona",
    *,
    fig_models: tuple[str, str],
    attr_short_labels: dict[str, str],
    n_full_participants: int,
    n_sub_participants: int,
) -> plt.Figure:
    """Plot a two-panel main-text figure for the Grummon et al. (2025) nutrition labeling RCT:
    Panel A: HC1 OLS treatment-effect coefficients and 95% CIs across the FOP arms (humans vs. the two `fig_models`).
    Panel B: Single-attribute choice cross-entropy (bits) across seen vs. unseen shopper attributes.
    """
    set_academic_palette(palette)
    by_name: dict[str, pd.DataFrame] = {}
    for key, df_m in model_ols_dfs.items():
        meta = parse_entry_metadata(key)
        if meta["persona"] == persona_mode:
            by_name[str(meta["model"])] = df_m

    # fig_models = (off-the-shelf display name, fine-tuned display name), e.g. ("audit-Kev 9B (base)", "audit-Kev 9B (FT)")
    kev_base, kev_ft = fig_models
    if kev_base not in by_name or kev_ft not in by_name:
        raise KeyError(f"Expected {kev_base} and {kev_ft} in {list(by_name.keys())}")

    # Order Panel A arms from largest to smallest Full Human treatment effect
    ordered_labels = list(human_ols_df["beta"].sort_values(ascending=False).index)
    short_arm_labels = [
        lbl.replace(" (vs. Positive)", "")
           .replace("Nutrition Info label", "Nutrition Info")
           .replace("High In warning label", "High In warning")
           .replace("Spectrum 1-to-5 star label", "Spectrum (1–5 stars)")
        for lbl in ordered_labels
    ]
    y_arms = np.arange(len(ordered_labels), dtype=np.float64)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.0, 4.9), dpi=150)
    fig.subplots_adjust(wspace=0.38)

    # Panel A: Forest plot of d=5 HC1 OLS treatment effects
    panel_a_specs: list[tuple[str, pd.DataFrame, str, str]] = [
        (f"Full human (N = {n_full_participants:,})", human_ols_df, MODEL_PALETTES[palette]["human"], "o"),
        (f"Human subsample (N = {n_sub_participants:,})", human_sub_ols_df, NEUTRALS["baseline"], "s"),
        (kev_base, by_name[kev_base], MODEL_PALETTES[palette]["off_the_shelf"], "^"),
        (kev_ft, by_name[kev_ft], MODEL_PALETTES[palette]["fine_tuned"], "D"),
    ]
    offsets_a = np.linspace(-0.24, 0.24, len(panel_a_specs))
    ax1.axvline(0.0, color=NEUTRALS["text"], linestyle="-", linewidth=1.4, zorder=2)
    for offset, (name, df_m, color, marker) in zip(offsets_a, panel_a_specs, strict=True):
        betas = df_m.loc[ordered_labels, "beta"].to_numpy(dtype=np.float64)
        ses = df_m.loc[ordered_labels, "se"].to_numpy(dtype=np.float64)
        ax1.errorbar(
            betas,
            y_arms + offset,
            xerr=1.96 * ses,
            fmt=marker,
            color=color,
            ecolor=color,
            elinewidth=1.2,
            capsize=2.3,
            markersize=5.4,
            label=name,
            zorder=3,
        )

    ax1.set_yticks(y_arms, labels=short_arm_labels, color=NEUTRALS["text"], fontsize=9.2)
    ax1.invert_yaxis()
    ax1.set_xlabel("HC1 OLS treatment effect vs. Positive arm β̂", color=NEUTRALS["muted"], fontsize=9.5)
    ax1.set_title("A. Causal treatment-effect estimates (5 arms)", color=NEUTRALS["text"], fontsize=10.5, fontweight="bold", loc="left", pad=8)
    ax1.tick_params(axis="x", colors=NEUTRALS["muted"], labelsize=9.0)
    ax1.grid(axis="x", color=NEUTRALS["grid"], linestyle="--", linewidth=0.8, zorder=0)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.spines["left"].set_color(NEUTRALS["muted"])
    ax1.spines["bottom"].set_color(NEUTRALS["muted"])
    ax1.legend(
        frameon=True,
        facecolor="white",
        edgecolor=NEUTRALS["legend_edge"],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=2,
        fontsize=8.5,
    )

    # Panel B: Seen vs. Unseen shopper attributes single-attribute choice cross-entropy (bits)
    # attr_short_labels maps attribute codes to axis labels; unseen attributes are marked with "*"
    attr_labels = [
        attr_short_labels[str(code)] + ("*" if grp == "Unseen in training" else "")
        for code, grp in zip(single_attr_ce_df["Code"], single_attr_ce_df["Attribute group"], strict=True)
    ]
    y_attrs = np.arange(len(attr_labels), dtype=np.float64)
    n_seen = int((single_attr_ce_df["Attribute group"] == "Seen in training").sum())
    n_unseen = len(single_attr_ce_df) - n_seen

    ce_base = single_attr_ce_df[kev_base].to_numpy(dtype=np.float64)
    ce_ft = single_attr_ce_df[kev_ft].to_numpy(dtype=np.float64)

    uniform_ce_bits = math.log2(3.0)
    ax2.axvline(uniform_ce_bits, color=NEUTRALS["ref_line"], linestyle=":", linewidth=1.1, zorder=1, label=f"Uniform 3-option ({uniform_ce_bits:.3f})")
    ax2.axhline(n_seen - 0.5, color=NEUTRALS["muted"], linestyle="--", linewidth=0.9, alpha=0.7, zorder=1)

    for idx in range(len(y_attrs)):
        ax2.plot(
            [ce_base[idx], ce_ft[idx]],
            [y_attrs[idx], y_attrs[idx]],
            color=NEUTRALS["baseline"],
            linewidth=1.5,
            zorder=2,
        )
    ax2.scatter(
        ce_base,
        y_attrs,
        marker="^",
        s=42,
        color=MODEL_PALETTES[palette]["off_the_shelf"],
        label=kev_base,
        zorder=3,
    )
    ax2.scatter(
        ce_ft,
        y_attrs,
        marker="D",
        s=36,
        color=MODEL_PALETTES[palette]["fine_tuned"],
        label=kev_ft,
        zorder=4,
    )

    ax2.set_yticks(y_attrs, labels=attr_labels, color=NEUTRALS["text"], fontsize=8.9)
    ax2.invert_yaxis()
    ax2.set_xlabel("Choice cross-entropy (bits; *unseen in training)", color=NEUTRALS["muted"], fontsize=9.5)
    ax2.set_title(f"B. Seen vs. unseen shopper attributes ({len(single_attr_ce_df)} attrs)", color=NEUTRALS["text"], fontsize=10.5, fontweight="bold", loc="left", pad=8)
    ax2.tick_params(axis="x", colors=NEUTRALS["muted"], labelsize=9.0)
    ax2.grid(axis="x", color=NEUTRALS["grid"], linestyle="--", linewidth=0.8, zorder=0)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.spines["left"].set_color(NEUTRALS["muted"])
    ax2.spines["bottom"].set_color(NEUTRALS["muted"])

    # Annotate Seen vs. Unseen regions cleanly on the right margin inside Panel B
    x_max = max(float(np.max(ce_base)), uniform_ce_bits) + 0.050
    x_min = float(np.min(ce_ft)) - 0.035
    ax2.set_xlim(x_min, x_max)
    ax2.text(
        x_max - 0.008,
        0.35,
        f"Seen in survey training ({n_seen})",
        ha="right",
        va="center",
        fontsize=8.0,
        fontstyle="italic",
        color=NEUTRALS["muted"],
    )
    ax2.text(
        x_max - 0.008,
        n_seen + 0.35,
        f"*Unseen in training ({n_unseen})",
        ha="right",
        va="center",
        fontsize=8.0,
        fontstyle="italic",
        color=NEUTRALS["muted"],
    )
    ax2.legend(
        frameon=True,
        facecolor="white",
        edgecolor=NEUTRALS["legend_edge"],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=2,
        fontsize=8.5,
    )

    plt.tight_layout()
    return fig

In [ ]:
fig_coefs = plot_grummon2025_coefs_and_attributes(
    human_ols_df,
    human_sub_ols_df,
    model_ols_dfs,
    single_attr_by_prefix[FIG_KEV_PREFIX],
    palette="nature",
    persona_mode="Respondent Persona",
    fig_models=tuple(FIG_KEV_PREFIX + m for m in FIG_MODELS),
    attr_short_labels=ATTR_SHORT_LABELS,
    n_full_participants=n_full_participants,
    n_sub_participants=n_sub_participants,
)
fig_coefs.savefig("grummon2025_ols_coefs.png", dpi=300, bbox_inches="tight")
fig_coefs.savefig("grummon2025_ols_coefs.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Bar groups for the error figures: one group per Kev size, each checkpoint followed by its run-set counterparts
ERROR_PLOT_GROUPS: dict[str, list[str]] = {}
for model_id in MODEL_IDS:
    display_name = format_raw_model_name(model_id)
    size = re.match(r"Kev [\d.]+B", display_name).group(0)
    ERROR_PLOT_GROUPS.setdefault(size, []).extend(prefix + display_name for prefix in prefixes)


def plot_grummon2025_error_comparison(
    comparison_metrics_df: pd.DataFrame,
    palette: str = "nature",
    persona_filter: str | None = None,
    *,
    bar_groups: dict[str, list[str]],
    n_sub_tasks: int,
) -> plt.Figure:
    """Plot a two-panel horizontal bar chart of MAPE (%) and MSE vs. Full Human beta*, bars grouped as in `bar_groups`.

    `bar_groups` maps a group title to model display names (top to bottom); groups are separated by a gap and a line,
    and the human subsample is drawn as its own group at the bottom. Each group gets its own palette color;
    off-the-shelf (base) bars are a light hatched tint of it and fine-tuned (FT) bars are solid.
    """
    pal = set_academic_palette(palette)
    group_palette = [pal[3], pal[4], pal[5], pal[1], pal[2]]  # pal[0] (navy) is reserved for the human subsample
    if len(bar_groups) > len(group_palette):
        raise ValueError(f"{len(bar_groups)} bar groups but only {len(group_palette)} group colors")

    df_plot = comparison_metrics_df.copy()
    if persona_filter is not None:
        mask = [
            ("human" in str(idx).lower()) or (persona_filter in str(idx))
            for idx in df_plot.index
        ]
        df_plot = df_plot.loc[mask]
        if df_plot.empty:
            raise ValueError(f"No rows in comparison_metrics_df matched persona_filter={persona_filter!r}")

    # Display name -> row key, for the model rows of this persona regime
    human_keys = [str(k) for k in df_plot.index if "human" in str(k).lower()]
    if len(human_keys) != 1:
        raise ValueError(f"Expected one human subsample row, got {human_keys}")
    key_by_name = {str(parse_entry_metadata(str(k))["model"]): str(k) for k in df_plot.index if str(k) not in human_keys}
    grouped_names = [name for names in bar_groups.values() for name in names]
    missing = [n for n in grouped_names if n not in key_by_name]
    if missing:
        raise KeyError(f"No rows for {missing} under {persona_filter!r}")
    ungrouped = sorted(set(key_by_name) - set(grouped_names))
    if ungrouped:
        raise ValueError(f"Rows not assigned to any bar group: {ungrouped}")

    # y positions: groups top to bottom with a gap between them, human subsample last
    group_gap = 0.7
    raw_models: list[str] = []
    y_list: list[float] = []
    bar_colors: list[str] = []
    bar_is_base: list[bool] = []
    separators: list[float] = []
    y = 0.0
    group_colors = dict(zip(bar_groups, group_palette))
    for group, names in [*bar_groups.items(), (None, [None])]:
        if raw_models:
            separators.append(y - 0.5 + group_gap / 2)
            y += group_gap
        for name in names:
            raw_models.append(human_keys[0] if name is None else key_by_name[name])
            y_list.append(y)
            bar_colors.append(MODEL_PALETTES[palette]["human"] if group is None else group_colors[group])
            bar_is_base.append(name is not None and "(base)" in name)
            y += 1.0
    y_pos = np.asarray(y_list)
    models = [format_entry_display_label(m, default_n=n_sub_tasks) for m in raw_models]

    mape_vals = df_plot.loc[raw_models, "MAPE vs. Full Human (%)"].to_numpy(dtype=np.float64)
    mse_vals = df_plot.loc[raw_models, "MSE vs. Full Human"].to_numpy(dtype=np.float64)

    fig_height = max(3.3, 0.56 * (y_pos[-1] + 1) + 0.9)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, fig_height), sharey=True, dpi=150)

    panel_specs = [
        (axes[0], mape_vals, "MAPE vs. full human β* (%)", "A. Mean absolute percentage error (MAPE)", "{:.1f}%"),
        (axes[1], mse_vals, "MSE vs. full human β*", "B. Mean squared error (MSE)", "{:.4f}"),
    ]
    for ax, vals, xlabel, title, fmt in panel_specs:
        bars = ax.barh(y_pos, vals, height=0.55, linewidth=0.8, zorder=3)
        for bar, color, is_base in zip(bars, bar_colors, bar_is_base, strict=True):
            if is_base:
                # Off-the-shelf: light tint of the group color with hatching in the full color
                bar.set_facecolor(tuple(0.35 * c + 0.65 for c in to_rgb(color)))
                bar.set_edgecolor(color)
                bar.set_hatch("////")
            else:
                bar.set_facecolor(color)
                bar.set_edgecolor(NEUTRALS["text"])
        ax.set_yticks(y_pos, labels=models, color=NEUTRALS["text"], fontsize=9.5)
        ax.set_ylim(y_pos[-1] + 0.6, y_pos[0] - 0.6)
        for sep in separators:
            ax.axhline(sep, color=NEUTRALS["baseline"], linewidth=0.8, zorder=1)
        ax.set_xlabel(xlabel, color=NEUTRALS["muted"], fontsize=9.5)
        ax.set_title(title, color=NEUTRALS["text"], fontsize=10.5, fontweight="bold", loc="left", pad=8)
        ax.tick_params(axis="x", colors=NEUTRALS["muted"], labelsize=9)
        ax.grid(axis="x", color=NEUTRALS["grid"], linestyle="--", linewidth=0.8, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color(NEUTRALS["muted"])
        ax.spines["bottom"].set_color(NEUTRALS["muted"])
        max_val = float(np.max(vals)) if len(vals) else 1.0
        ax.set_xlim(0, max_val * 1.24)
        for bar, val in zip(bars, vals, strict=True):
            ax.text(
                val + max_val * 0.02,
                bar.get_y() + bar.get_height() / 2,
                fmt.format(val),
                va="center",
                ha="left",
                fontsize=9,
                color=NEUTRALS["muted"],
                fontweight="bold",
            )

    style_handles = [
        Patch(facecolor=tuple(0.35 * c + 0.65 for c in to_rgb(NEUTRALS["muted"])), edgecolor=NEUTRALS["muted"], hatch="////", label="Off-the-shelf (base)"),
        Patch(facecolor=NEUTRALS["muted"], edgecolor=NEUTRALS["text"], label="Fine-tuned (FT)"),
    ]
    fig.legend(
        handles=style_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=2,
        frameon=True,
        facecolor="white",
        edgecolor=NEUTRALS["legend_edge"],
        fontsize=9,
    )
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    return fig

In [ ]:
for mode in PERSONA_MODES:
    fig_err = plot_grummon2025_error_comparison(
        comparison_metrics_df,
        palette="nature",
        persona_filter=PERSONA_LABELS[mode],
        bar_groups=ERROR_PLOT_GROUPS,
        n_sub_tasks=n_sub_tasks,
    )
    fig_err.savefig(f"grummon2025_ols_{mode}.png", dpi=300, bbox_inches="tight")
    fig_err.savefig(f"grummon2025_ols_{mode}.pdf", bbox_inches="tight")
    plt.show()